In [11]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
# -----------------------
# LOAD DATA
# -----------------------
df = pd.read_csv("books.csv", delimiter=',')

# Basic inspection
print(df.shape)
print(df.columns)
df.head()

(6810, 12)
Index(['isbn13', 'isbn10', 'title', 'subtitle', 'authors', 'categories',
       'thumbnail', 'description', 'published_year', 'average_rating',
       'num_pages', 'ratings_count'],
      dtype='object')


,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


In [13]:
df = df[['title', 'authors', 'categories', 'description']]
print(df.shape)
df.head()

(6810, 4)


,title,authors,categories,description
0,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...
1,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,A new 'Christie for Christmas' -- a full-lengt...
2,The One Tree,Stephen R. Donaldson,American fiction,Volume Two of Stephen Donaldson's acclaimed se...
3,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b..."
4,The Four Loves,Clive Staples Lewis,Christian life,Lewis' work on the nature of love divides love...


In [14]:
# Fill missing values
df = df.fillna('')

# -----------------------
# TEXT CLEANING
# -----------------------
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text
    # re = regular expressions. It is a Python library for pattern matching in text.
    # Find anything that is NOT a lowercase letter or a space: a-z → all lowercase letters, \s → spaces, [^ ... ] → NOT these characters
    # So find: punctuation (! , . ?), numbers (1, 2, 3), symbols (#, @, %)


In [15]:
# -----------------------
# CONCATENATE FIELDS
# -----------------------
df['text'] = (
    df['title'] + ' ' +
    df['authors'] + ' ' +
    df['categories'] + ' ' +
    df['description']
)

df['text'] = df['text'].apply(clean_text)


In [16]:
# -----------------------
# BAG OF WORDS
# -----------------------
bow_vectorizer = CountVectorizer(max_features=5000, stop_words='english') # limit vocabulary size; # remove common words. Examples removed: the, and, is, in, of
X_bow = bow_vectorizer.fit_transform(df['text'])

bow_similarity = cosine_similarity(X_bow)

# stop words removal is NOT the same as TF-IDF.
#   Stop words = remove some words completely
#   TF-IDF = keep all words, but reduce the importance of common ones


In [17]:
# -----------------------
# TF-IDF
# -----------------------
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf_vectorizer.fit_transform(df['text'])

tfidf_similarity = cosine_similarity(X_tfidf)


In [18]:
# -----------------------
# BERT / TRANSFORMERS EMBEDDINGS
# -----------------------

from transformers import AutoTokenizer, AutoModel
import torch

# Load pretrained model (lightweight and good default)
model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Put model in eval mode
model.eval()


# -----------------------
# FUNCTION: GET EMBEDDING
# -----------------------
def get_embedding(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # Mean pooling (standard way to get sentence embedding)
    embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings.squeeze().numpy()


# -----------------------
# COMPUTE ALL BOOK EMBEDDINGS
# -----------------------
bert_embeddings = np.vstack([
    get_embedding(text) for text in df['text'].tolist()
])

# -----------------------
# SIMILARITY MATRIX
# -----------------------
bert_similarity = cosine_similarity(bert_embeddings)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
# -----------------------
# RECOMMENDER FUNCTION
# -----------------------
def recommend(query, top_k=5, model_type='bert'):

    query_clean = clean_text(query)

    # -----------------------
    # TF-IDF
    # -----------------------
    if model_type == 'tfidf':

        query_vec = tfidf_vectorizer.transform([query_clean])
        scores = cosine_similarity(query_vec, X_tfidf)[0]

    # -----------------------
    # BOW
    # -----------------------
    elif model_type == 'bow':

        query_vec = bow_vectorizer.transform([query_clean])
        scores = cosine_similarity(query_vec, X_bow)[0]

    # -----------------------
    # BERT
    # -----------------------
    elif model_type == 'bert':

        query_vec = get_embedding(query_clean).reshape(1, -1)
        scores = cosine_similarity(query_vec, bert_embeddings)[0]

    else:
        raise ValueError("model_type must be: 'tfidf', 'bow', or 'bert'")

    # -----------------------
    # TOP-K RESULTS
    # -----------------------
    top_idx = np.argsort(scores)[::-1][1:top_k+1]

    return df.iloc[top_idx][['title', 'authors', 'categories', 'description']]

In [29]:
recommend("Book about computers and science", model_type='bow')

,title,authors,categories,description
3645,2061,Arthur C. Clarke,Computers,Science fiction-roman.
1364,"Turtles, Termites, and Traffic Jams",Mitchel Resnick,Computers,"""A Bradford book."" Includes bibliographical re..."
5054,The Anatomy Coloring Book,Wynn Kapit;Lawrence M. Elson,Science,Includes bibliographical references and index
6517,The Science Book,Peter Tallack,Science,This chronology of scientific discoveries and ...
3712,Unweaving the Rainbow,Richard Dawkins,Science,"Offers an assessment of what science is, how i..."
